In [1]:
import pandas as pd
import networkx as nx
from ipysigma import Sigma


In [2]:
artists = pd.read_parquet("D:/data/spotify_clean_parquet/artists.parquet")
# artist_albums = pd.read_parquet("D:/data/spotify_clean_parquet/artist_albums.parquet")
# albums = pd.read_parquet("D:/data/spotify_clean_parquet/albums.parquet")

tracks = pd.read_parquet("D:/data/spotify_clean_parquet/tracks.parquet", columns=['rowid', 'id', 'name', 'popularity'])
artist_tracks = pd.read_parquet("D:/data/spotify_clean_parquet/track_artists.parquet")

In [3]:
reduced_tracks = tracks.loc[tracks['popularity'] > 10]
reduced_artist_tracks = artist_tracks.loc[artist_tracks['track_rowid'].isin(reduced_tracks['rowid'].to_list())]
# print(reduced_tracks)

In [4]:
print(artist_tracks[artist_tracks.columns[0]].count())
print(reduced_artist_tracks[reduced_artist_tracks.columns[0]].count())

print(tracks[tracks.columns[0]].count())
print(reduced_tracks[reduced_tracks.columns[0]].count())



348055756
16007275
256039007
11420991


In [ ]:
# artist = artists.loc[artists['id'] == '2J9bayqEs1vgpZWed9pHgH'] # bat ventsi
# artist = artists.loc[artists['id'] == '699OTQXzgjhIYAHMy9RyPD'] # Carti
# artist = artists.loc[artists['id'] == '1FAE6aKS3Tgrqpiw2fQbj4'] # Miro
# artist = artists.loc[artists['id'] == '1vAwQYTE1k5MBhNsvqphp1'] # Azis
# artist = artists.loc[artists['id'] == '78rUTD7y6Cy67W1RVzYs7t'] # PinkPantheress
# artist = artists.loc[artists['id'] == '2tRsMl4eGxwoNabM08Dm4I'] # Judas Priest
# artist = artists.loc[artists['id'] == '1eSLQvCYcfd5f1jgxBGbKJ'] # 100 Kila
artist = artists.loc[artists['id'] == '0du5cEVh5yTK9QJze8zA0C'] # Bruno Mars

print(artist)

           rowid                      id     fetched_at        name  \
6480653  6480654  0du5cEVh5yTK9QJze8zA0C  1743638400000  Bruno Mars   

         followers_total  popularity  
6480653         70128274          95  


In [5]:
G = nx.Graph()
labels = dict()
visited = list()

In [6]:
def find_collabs_pop10(artist_rowid):
    if artist_rowid in visited:
        return
    visited.append(artist_rowid)
    labels[artist_rowid] = artists.loc[artists['rowid'] == artist_rowid]['name'].to_list()[0]

    all_tracks = reduced_artist_tracks.loc[reduced_artist_tracks['artist_rowid'] == artist_rowid]['track_rowid']

    collabs = reduced_artist_tracks.loc[(reduced_artist_tracks['track_rowid'].isin(all_tracks.to_list())) & (reduced_artist_tracks['artist_rowid'] != artist_rowid)]['artist_rowid']
    # print(collabs)

    
    # collabs_dict = artists.loc[artists['rowid'].isin(collabs)]
    for collaborator in collabs.to_list():
        if not G.has_node(artist_rowid):
            G.add_node(artist_rowid)
        if not G.has_edge(artist_rowid, collaborator):
            G.add_edge(artist_rowid, collaborator)
        # print(collabs_dict.loc[collabs_dict['rowid'] == collaborator]['name'].to_list()[0])
        # labels[collaborator] = collabs_dict.loc[collabs_dict['rowid'] == collaborator]['name'].to_list()[0]
    print(f'\r {G.number_of_nodes()}', end='')
    return collabs.to_list()

In [99]:
def find_collabs(artist_rowid):
    if not G.has_node(artist_rowid):
        G.add_node(artist_rowid)
    labels[artist_rowid] = artists.loc[artists['rowid'] == artist_rowid]['name'].to_list()[0]

    all_tracks = artist_tracks.loc[artist_tracks['artist_rowid'] == artist_rowid]['track_rowid']

    collabs = artist_tracks.loc[(artist_tracks['track_rowid'].isin(all_tracks.to_list())) & (artist_tracks['artist_rowid'] != artist_rowid)]['artist_rowid']
    # print(collabs)

    # print(artist_tracks.loc[(artist_tracks['track_rowid'].isin(all_tracks.to_list())) & (artist_tracks['artist_rowid'] != artist_rowid)])
    
    
    collabs_dict = artists.loc[artists['rowid'].isin(collabs)]
    for collaborator in collabs.to_list():
        if not G.has_node(artist_rowid):
            G.add_node(artist_rowid)
        if not G.has_edge(artist_rowid, collaborator):
            G.add_edge(artist_rowid, collaborator)
        # print(collabs_dict.loc[collabs_dict['rowid'] == collaborator]['name'].to_list()[0])
        labels[collaborator] = collabs_dict.loc[collabs_dict['rowid'] == collaborator]['name'].to_list()[0]
    return collabs.to_list()

In [7]:
# todo = find_collabs(4988673)
artist_rowid = 6480654


G.add_node(artist_rowid)
first = find_collabs_pop10(artist_rowid)
for ar_f in first:
    second = find_collabs_pop10(ar_f)
    if second is not None:
        for ar_s in second:
            third = find_collabs_pop10(ar_s)
    #         if third is not None:
    #             for ar_t in third:
    #                 find_collabs_pop10(ar_t)

 49637

In [ ]:
nx.draw(G, with_labels=True, labels=labels)

Grab a graph and relabel all the nodes // not yet working

In [ ]:
G = nx.read_gexf('brunoMars3.gexf')

In [18]:
for node in list(G.nodes()):
    visited.append(int(node))
# str_to_int = dict(zip(list(G.nodes()), visited))
# nx.relabel_nodes(G, str_to_int)

# labels_df = artists.loc[artists['rowid'].isin(visited)]
# labels = labels_df.set_index('rowid')['name'].to_dict()

# print(labels)

Add followers to each node

In [19]:
labels_df = artists.loc[artists['rowid'].isin(visited)]
followers_total = labels_df.set_index('rowid')['followers_total'].to_dict()
nx.set_node_attributes(G, followers_total, "followers")
print(followers_total)

{39: 9503, 107: 5485, 292: 287, 522: 444625, 642: 1325, 800: 1302, 876: 4045, 1128: 1196, 1163: 174, 1233: 23999, 1768: 168, 1805: 123718, 2051: 9444, 2601: 2418, 2604: 378, 2730: 22829, 2891: 317511, 3060: 5960, 3084: 387042, 3146: 116709, 3172: 130, 3282: 5881, 3286: 1488, 3315: 342442, 3353: 53528, 3365: 22798, 3545: 3463, 3636: 159830, 3910: 276, 3917: 55703, 4133: 358, 4210: 3128, 4263: 36604, 4318: 399, 4349: 25452, 4479: 513, 5010: 842, 5090: 719, 5356: 33881, 5773: 130721, 5956: 3127665, 6287: 441784, 6370: 19562, 6613: 408, 6914: 7336, 7139: 419082, 7177: 191, 7255: 17, 7620: 102, 7751: 870, 7988: 1872, 8013: 3073, 8168: 6382, 8362: 33770, 8487: 2362, 8605: 13524, 8733: 519947, 8775: 202845, 9129: 10605, 9256: 30060, 9335: 267, 9484: 17101, 9511: 10649, 9534: 7171, 9846: 2313, 9869: 10, 9955: 1082761, 10169: 216553, 10273: 1256, 10576: 2514, 10766: 64, 10968: 963, 10988: 6240, 11371: 2168, 11454: 59013, 11466: 197735, 11615: 5397, 11658: 92476, 11993: 99284, 12207: 136869, 124

In [20]:
labels_df = artists.loc[artists['rowid'].isin(visited)]
names = labels_df.set_index('rowid')['name'].to_dict()
H = nx.relabel_nodes(G, names)

In [21]:
print(G.number_of_nodes())
print(H.number_of_nodes())

49637
47839


In [ ]:
# H = nx.relabel_nodes(G, labels)
nx.write_gexf(G, "test.gexf")

In [77]:
P = nx.read_gexf('pink4.gexf')
ids = list()
for id in P.nodes():
    ids.append(int(id))
labels_df = artists.loc[artists['rowid'].isin(ids)]

labels = dict()
lls = labels_df.set_index('rowid')['name'].to_dict()
for k, v in lls.items():
    labels[str(k)] = v
# labels = dict(zip(labels['rowid'].astype(str), labels['name']))

H = nx.relabel_nodes(P, labels)
nx.write_gexf(H, "test.gexf")

In [22]:
unweighted_path = nx.shortest_path(H, source="Steve Vai", target="Bruno Mars")
# unweighted_path = nx.shortest_path(G, source=5333435, target=3766345)
print(unweighted_path)

['Steve Vai', 'Eros Ramazzotti', 'Rhythms Del Mundo', 'Bruno Mars']


In [92]:
# for name, age in names.items():  # for name, age in dictionary.iteritems():  (for Python 2.x)
#     if age == "Michael Jackson":
#         print(name)
print(names[269399])
# print(nx.degree_histogram(G))
haha = dict(G.degree)
skibidi = dict(sorted(haha.items(), key=lambda item: item[1]))
# print(skibidi)

Snoop Dogg


In [23]:
Sigma(H, node_size=nx.get_node_attributes(H, 'followers'), node_color='category')


Sigma(nx.Graph with 47,839 nodes and 117,841 edges)